[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/natrask/AESCAPE/blob/main/notebooks/05_mcp.ipynb)

# Part 5 — MCP

**MCP is the Model Context Protocol**: an open standard for how an LLM application discovers
and calls tools that live outside its own process.

In Parts 1–4 every tool was a Python function in the same process as the agent loop. That is
fine for a notebook and wrong for a real simulation stack, where the solver is a compiled
binary on a cluster behind a scheduler, written by someone else.

MCP splits that in two:

- a **server** owns the tools and publishes their names, descriptions and JSON schemas;
- a **client** connects, asks what is available, and calls what it needs.

Write your solver's tool surface once as a server, and anything that speaks MCP can drive it —
this notebook today, a colleague's pipeline tomorrow, a desktop assistant after that. The
server does not know or care which.

In [ ]:
import sys
if 'google.colab' in sys.modules:
    %pip install -U -q mcp scikit-fem

import os, json, asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

print("mcp client ready.")

## The server

A server is a plain Python file. `@mcp.tool()` publishes a function: the **type hints become
the JSON schema** and the **docstring becomes the description** the model reads.

Two tools below — a calculator, and a scikit-fem solve of

$$-\Delta u = 1 \quad \text{in } \Omega, \qquad u = 0 \quad \text{on } \partial\Omega$$

on the L-shaped domain. Note the second one returns numbers, not a mesh: everything crossing
the boundary has to be JSON.

In [ ]:
%%writefile aescape_mcp_server.py
from mcp.server.mcpserver import MCPServer

import numpy as np
from skfem import (MeshTri, Basis, ElementTriP1, BilinearForm, LinearForm,
                   condense, solve)
from skfem.helpers import dot, grad

mcp = MCPServer(name="aescape-tools")


@BilinearForm
def stiffness(u, v, w):
    return dot(grad(u), grad(v))


@LinearForm
def unit_load(v, w):
    return 1.0 * v


@mcp.tool()
def calculator(op: str, a: float, b: float) -> dict:
    """Perform one arithmetic operation. op is one of: add, sub, mul, div."""
    ops = {"add": a + b, "sub": a - b, "mul": a * b,
           "div": a / b if b != 0 else float("nan")}
    if op not in ops:
        return {"error": f"unknown op {op!r}; valid: add, sub, mul, div"}
    return {"result": ops[op]}


@mcp.tool()
def solve_poisson(refine: int = 3) -> dict:
    """Solve -laplace(u) = 1 on the L-shaped domain with u = 0 on the boundary.

    refine: number of uniform refinements, 0 to 6.
    """
    if not 0 <= refine <= 6:
        return {"error": f"refine must be between 0 and 6; got {refine}"}
    mesh = MeshTri.init_lshaped()
    for _ in range(refine):
        mesh = mesh.refined()
    basis = Basis(mesh, ElementTriP1())
    K = stiffness.assemble(basis)
    f = unit_load.assemble(basis)
    u = solve(*condense(K, f, D=mesh.boundary_nodes()))
    return {"refine": refine,
            "dofs": int(basis.N),
            "elements": int(mesh.nelements),
            "u_max": float(np.max(u))}


if __name__ == "__main__":
    mcp.run(transport="stdio")

## The client

`stdio` transport launches the server as a subprocess and talks to it over stdin/stdout.
(There is also `streamable-http` for a server on another machine.)

`list_tools()` is the discovery step — the client learns the tool surface at runtime rather
than having it hard-coded. That is the whole point of the protocol.

In [ ]:
async def demo():
    params = StdioServerParameters(command=sys.executable, args=["aescape_mcp_server.py"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            listed = await session.list_tools()
            print(f"discovered {len(listed.tools)} tools\n")
            for t in listed.tools:
                schema = t.input_schema
                print(f"  {t.name}")
                print(f"    {(t.description or '').splitlines()[0]}")
                print(f"    params   {list(schema.get('properties', {}))}")
                print(f"    required {schema.get('required', [])}\n")

            r = await session.call_tool("calculator", {"op": "mul", "a": 13, "b": 47})
            print("calculator(mul, 13, 47) ->", r.content[0].text)

            r = await session.call_tool("solve_poisson", {"refine": 4})
            print("solve_poisson(refine=4)  ->", r.content[0].text)

            # guardrails live in the tool, so a bad argument comes back as data
            r = await session.call_tool("solve_poisson", {"refine": 99})
            print("solve_poisson(refine=99) ->", r.content[0].text)

await demo()

## Syntax summary

**Server**

| | |
|---|---|
| `from mcp.server.mcpserver import MCPServer` | the server class |
| `mcp = MCPServer(name="...")` | create it |
| `@mcp.tool()` | publish a function; hints → schema, docstring → description |
| `mcp.run(transport="stdio")` | serve over stdin/stdout (or `"streamable-http"`) |

**Client**

| | |
|---|---|
| `StdioServerParameters(command=..., args=[...])` | how to launch the server |
| `async with stdio_client(params) as (read, write)` | open the transport |
| `async with ClientSession(read, write) as session` | open a session |
| `await session.initialize()` | handshake |
| `await session.list_tools()` | discover — `.tools[i].name`, `.description`, `.input_schema` |
| `await session.call_tool(name, {...})` | invoke — result in `.content[0].text` |

Two things that will bite you:

- **The SDK renamed things in 2.x.** `FastMCP` is now `MCPServer`, and `inputSchema` is now
  `input_schema`. Most tutorials online are still 1.x. Pin `mcp<2` if you need the old API.
- **Never `print()` inside a stdio tool.** stdout *is* the protocol channel; a stray print
  corrupts the stream. Log to stderr instead.

## Further reading

- [modelcontextprotocol.io](https://modelcontextprotocol.io) — specification and concepts
- [Python SDK](https://github.com/modelcontextprotocol/python-sdk) — source and examples
- [2.x migration guide](https://py.sdk.modelcontextprotocol.io/v2/migration/) — what changed from 1.x
- [Reference servers](https://github.com/modelcontextprotocol/servers) — filesystem, git, databases and more